# Objectives
Fraud Detection Machine Learning Pipeline (using Amazon SageMaker). The followig services are used in the project:

* **Amazon SageMaker Pipelines** → orchestration
* **SageMaker Training Jobs** → XGBoost training
* **SageMaker Processing Jobs** → preprocessing/evaluation
* **SageMaker Model Registry** → versioned models
* **CodePipeline + CodeBuild** → CI/CD
* **SageMaker Endpoint** → production inference

# Directory Structure

In [1]:
%%writefile README.md
# Project Structure
fraud-sagemaker-mlops/
│
├── README.md
├── requirements.txt
│
├── config.py
│
├── pipeline.py
│
├── scripts/
│   │
│   ├── preprocess.py
│   ├── train.py
│   ├── evaluate.py
│   └── inference.py
│
├── deployment/
│   └── deploy_endpoint.py
│
└── data/
    └── raw
        └── fraud_transactions_dataset.csv

Writing README.md


In [2]:
# ---------------------
# Create the Directory Structure
# ---------------------
!mkdir -p data scripts deployment

In [3]:
%%writefile config.py
AWS_REGION = "us-east-1"

ROLE_ARN = (
    "arn:aws:iam::767398056079:role/cfst-4216-4b2b48c80a3e12e27a-SageMakerExecutionRole-R7DkiJ7J7K47"
) # Make sure the ARN is in the form arn:aws:iam::<account-id>:role/<role-name>

BUCKET = "sagemaker-us-east-1-767398056079"

PIPELINE_NAME = "fraud-detection-xgboost-pipeline"

MODEL_PACKAGE_GROUP = "fraud-detection-models"

ENDPOINT_NAME = "fraud-detection-production"

Writing config.py


In [4]:
%%writefile requirements.txt
sagemaker==2.248.0
boto3
pandas
numpy
scikit-learn
xgboost

Writing requirements.txt


In [ ]:
!pip install -r requirements.txt

# Preprocessing

In [6]:
%%writefile scripts/preprocess.py
# import numpy as np

# from sklearn.preprocessing import LabelEncoder

# from xgboost import XGBClassifier

# import joblib

import argparse
import os

import pandas as pd
from sklearn.model_selection import train_test_split
def preprocess(input_file, output_dir):
    print(f"Reading input file: {input_file}")

    df = pd.read_csv(input_file)

    print("Dataset Preview:")
    print(df.head())

    
    # Feature engineering
   
    df["transaction_time"] = pd.to_datetime(
        df["transaction_time"]
    )

    df["hour"] = df["transaction_time"].dt.hour
    df["day"] = df["transaction_time"].dt.day
    df["month"] = df["transaction_time"].dt.month


    # Remove identifiers and the redundant transaction_time
    
    df.drop(
        [
            "transaction_id",
            "customer_id",
            "transaction_time"
        ],
        axis=1,
        inplace=True
    )

   
    # Convert categorical Yes/No
   
    df["card_present"] = df["card_present"].map(
        {
            "Yes": 1,
            "No": 0
        }
    )

    df["previous_fraud"] = df["previous_fraud"].map(
        {
            "Yes": 1,
            "No": 0
        }
    )


    # One-hot encoding
  
    df = pd.get_dummies(
        df,
        columns=[
            "merchant_category",
            "transaction_country"
        ]
    )

    # Make sure the output directory exists
    os.makedirs(
        output_dir,
        exist_ok=True
    )

    train_df, test_df = train_test_split(
        df,
        test_size=0.2,
        random_state=42,
        stratify=df["fraud"]
    )
    
    # output_file = os.path.join(
    #     output_dir,
    #     "train.csv"
    # )

    # df.to_csv(
    #     output_file,
    #     index=False
    # )

    train_df.to_csv(
        f"{output_dir}/train.csv",
        index=False
    )

    test_df.to_csv(
        f"{output_dir}/test.csv",
        index=False
    )
    
    #print(f"Processed data written to: {output_file}")
    print(f"Processed shape: {df.shape}")

if __name__ == "__main__":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--input",
        required=True
    )

    parser.add_argument(
        "--output",
        required=True
    )

    args = parser.parse_args()

    preprocess(
        args.input,
        args.output
    )

Writing scripts/preprocess.py


# Training

In [7]:
%%writefile scripts/train.py

import argparse
import pandas as pd
import xgboost as xgb
import os

def train(
    train_path,
    model_path
):

    df = pd.read_csv(
        train_path
    )

    X = df.drop(
        "fraud",
        axis=1
    )

    y = df.fraud

    model = xgb.XGBClassifier(
        n_estimators=300,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="auc",
        scale_pos_weight=5
    )

    model.fit(
        X,
        y
    )

    os.makedirs(
        model_path,
        exist_ok=True
    )

    model.save_model(
        f"{model_path}/model.json"
    )

    
    #print(f"Model saved to: {model_file}")
    print(f"Model saved to: {model_path}/model.json")

if __name__ == "__main__":

    parser = argparse.ArgumentParser() 
    
    parser.add_argument(
        "--train",
        required=True
    )

    parser.add_argument(
        "--model-dir",
        required=True
    )

    args = parser.parse_args()

    train(
        args.train,
        args.model_dir
    )

Writing scripts/train.py


# Evaluate

In [8]:
%%writefile scripts/evaluate.py
import argparse
import json

import pandas as pd
import xgboost as xgb

from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score
    )

# from sklearn.metrics import (
#     accuracy_score,
#     precision_score,
#     recall_score,
#     f1_score,
#     classification_report,
#     roc_auc_score
# )

def evaluate(
    model_path,
    test_path,
    output_path
    ):

    model = xgb.XGBClassifier()

    model.load_model(
        model_path
    )

    df = pd.read_csv(
        test_path
    )

    X = df.drop(
        "fraud",
        axis=1
    )

    y = df["fraud"]

    prediction = model.predict(
        X
    )

    probability = model.predict_proba(
        X
    )[:, 1]

    metrics = {
        "auc": roc_auc_score(
            y,
            probability
        ),

        "precision": precision_score(
            y,
            prediction,
            zero_division=0
        ),

        "recall": recall_score(
            y,
            prediction,
            zero_division=0
        )
    }

    print("Evaluation metrics:")
    print(metrics)

    with open(
        output_path,
        "w"
    ) as f:

        json.dump(
            metrics,
            f,
            indent=2
        )

    print(
        f"Metrics written to: {output_path}"
    )


if name == "main":

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--model",
        required=True
    )

    parser.add_argument(
        "--test",
        required=True
    )

    parser.add_argument(
        "--output",
        required=True
    )

    args = parser.parse_args()

    evaluate(
        args.model,
        args.test,
        args.output
    )

Writing scripts/evaluate.py


# SageMaker Pipeline

In [9]:
%%writefile pipeline.py
import sagemaker

from sagemaker.xgboost.estimator import XGBoost

from sagemaker.workflow.pipeline import Pipeline

from sagemaker.workflow.steps import (
    ProcessingStep,
    TrainingStep
)

from sagemaker.processing import (
    ScriptProcessor,
    ProcessingInput,
    ProcessingOutput
)

from sagemaker.estimator import Estimator

from sagemaker.inputs import TrainingInput

from sagemaker.workflow.functions import Join

import config


#SageMaker session
session = sagemaker.Session()

#Processing
processor = ScriptProcessor(
    image_uri=
    sagemaker.image_uris.retrieve(
        framework="sklearn",
        region=session.boto_region_name,
        version="1.2-1"
    ),

    command=[
        "python3"
    ],

    role=config.ROLE_ARN,

    instance_count=1,

    instance_type="ml.m5.large"

)

processing_step = ProcessingStep(
    name="FraudDataProcessing",
    processor=processor,

    inputs=[
        ProcessingInput(
            source=f"s3://{config.BUCKET}/raw/fraud_transactions_dataset.csv",
            destination="/opt/ml/processing/input"
            )
        ],
    outputs=[
        ProcessingOutput(
            output_name="processed_data",
            source="/opt/ml/processing/output",
            destination=f"s3://{config.BUCKET}/processed/"
        )
    ],
    code=
    "scripts/preprocess.py",
    job_arguments=[
        "--input",
        "/opt/ml/processing/input/fraud_transactions_dataset.csv",

        "--output",
        "/opt/ml/processing/output"
    ]
)

# # XGBoost Training
# xgb_estimator = Estimator(
#     image_uri=
#     sagemaker.image_uris.retrieve(
#         framework="xgboost",
#         region=session.boto_region_name,
#         version="1.7-1"
#     ),

#     role=config.ROLE_ARN,
#     instance_count=1,
#     instance_type="ml.m5.large",
#     output_path=
#     f"s3://{config.BUCKET}/models"
# )

xgb_estimator = XGBoost(
    entry_point="train.py",
    source_dir="scripts",
    framework_version="1.7-1",
    role=config.ROLE_ARN,
    instance_count=1,
    instance_type="ml.m5.large",
    output_path=f"s3://{config.BUCKET}/models",
    hyperparameters={
        "train": "/opt/ml/input/data/train/train.csv",
        "model-dir": "/opt/ml/model"
    }
)


# training_step = TrainingStep(
#     name="FraudXGBoostTraining",
#     estimator=xgb_estimator,
#     inputs={
#         "train": TrainingInput(
#             s3_data=processing_step.properties
#                 .ProcessingOutputConfig
#                 .Outputs["processed_data"]
#                 .S3Output
#                 .S3Uri,

#             content_type="text/csv"
#         )
#     }
# )

training_step = TrainingStep(
    name="FraudXGBoostTraining",
    estimator=xgb_estimator,
    inputs={
        "train": TrainingInput(
            s3_data=Join(
                on="",
                values=[
                    processing_step.properties
                        .ProcessingOutputConfig
                        .Outputs["processed_data"]
                        .S3Output
                        .S3Uri,
                    "train.csv"
                ]
            ),
            content_type="text/csv"
        )
    }
)

evaluation_processor = ScriptProcessor(

    image_uri=sagemaker.image_uris.retrieve(
        framework="xgboost",
        region=session.boto_region_name,
        version="1.7-1"
    ),

    command=["python3"],

    role=config.ROLE_ARN,

    instance_count=1,

    instance_type="ml.m5.large"
)

evaluation_step = ProcessingStep(

    name="FraudModelEvaluation",

    processor=evaluation_processor,

    inputs=[

        ProcessingInput(

            source=training_step
                .properties
                .ModelArtifacts
                .S3ModelArtifacts,

            destination="/opt/ml/processing/model"
        ),

        ProcessingInput(

            source=processing_step
                .properties
                .ProcessingOutputConfig
                .Outputs["processed_data"]
                .S3Output
                .S3Uri,

            destination="/opt/ml/processing/test"
        )
    ],

    outputs=[

        ProcessingOutput(

            output_name="evaluation",

            source="/opt/ml/processing/evaluation",

            destination=f"s3://{config.BUCKET}/evaluation/"
        )
    ],

    code="scripts/evaluate.py",

    job_arguments=[

        "--model",
        "/opt/ml/processing/model/model.tar.gz",

        "--test",
        "/opt/ml/processing/test/test.csv",

        "--output",
        "/opt/ml/processing/evaluation/evaluation.json"
    ]
)

# Pipeline
pipeline = Pipeline(
    name=config.PIPELINE_NAME,
    steps=[
        processing_step,
        training_step
    ]
)

#Create / update pipeline
pipeline.upsert(
    role_arn=config.ROLE_ARN
)

print("Pipeline created/updated successfully.")

Writing pipeline.py


# Run the pipeline
Inside SageMaker Studio, run:

In [10]:
!python pipeline.py

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
INFO:sagemaker.image_uris:Defaulting to only available Python version: py3
INFO:sagemaker.image_uris:Defaulting to only supported image scope: cpu.
INFO:sagemaker.image_uris:Ignoring unnecessary Python version: py3.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: ml.m5.large.
INFO:sagemaker.image_uris:Ignoring unnecessary instance type: None.
INFO:sagemaker:Created S3 bucket: sagemaker-us-east-1-767398056079
Pipeline created/updated successfully.


# Verify if the pipeline is created

Open AWS Console -> SageMaker Studio -> Pipelines -> fraud-detection-xgboost-pipeline

Now you can execute it by clicking "Execute Latest Version" or you can schedule its execution.

# Closing
Congratulations! This is a key milestone that concludes the pipeline creation in SageMaker! 

---

# Prediction Endpoint

In [13]:
%%writefile scripts/inference.py
import os
import io
import json

import numpy as np
import pandas as pd
import xgboost as xgb


def model_fn(model_dir):
    """
    Load the XGBoost model from the SageMaker model directory.
    """

    model_path = os.path.join(model_dir, "model.json")

    model = xgb.XGBClassifier()
    model.load_model(model_path)

    return model


def input_fn(request_body, request_content_type):
    """
    Convert incoming request into a pandas DataFrame.

    Expected input:

        0.12,0.45,1.0,23.0,...
        0.20,0.32,0.0,17.0,...

    CSV must NOT contain the 'fraud' target column.
    """

    if request_content_type == "text/csv":

        if isinstance(request_body, bytes):
            request_body = request_body.decode("utf-8")

        data = pd.read_csv(
            io.StringIO(request_body),
            header=None
        )

        return data

    elif request_content_type == "application/json":

        data = json.loads(request_body)

        return pd.DataFrame(data)

    else:
        raise ValueError(
            f"Unsupported content type: {request_content_type}"
        )


def predict_fn(input_data, model):
    """
    Run prediction.

    Returns:
        fraud_probability
        fraud_prediction
    """

    probabilities = model.predict_proba(input_data)[:, 1]

    predictions = (probabilities >= 0.5).astype(int)

    return {
        "fraud_probability": probabilities.tolist(),
        "fraud_prediction": predictions.tolist()
    }


def output_fn(prediction, accept):
    """
    Serialize prediction result.
    """

    if accept == "application/json" or accept == "*/*":

        return (
            json.dumps(prediction),
            "application/json"
        )

    raise ValueError(
        f"Unsupported accept type: {accept}"
    )

Writing scripts/inference.py


In [14]:
%%writefile endpoint.py
from sagemaker.xgboost import XGBoostModel
import config 

from sagemaker.serializers import CSVSerializer
from sagemaker.deserializers import JSONDeserializer

model_artifact_s3_uri = "s3://sagemaker-us-east-1-767398056079/models/pipelines-ubqscfj9noh2-FraudXGBoostTraining-QsPceugO0I/output/model.tar.gz"
xgb_model = XGBoostModel(
    model_data=model_artifact_s3_uri,
    role=config.ROLE_ARN,
    entry_point="inference.py",
    source_dir="scripts",
    framework_version="1.7-1",
    py_version="py3"
)

predictor = xgb_model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="fraud-xgboost-endpoint"
)

predictor.serializer = CSVSerializer()
predictor.deserializer = JSONDeserializer()

features = [
    12.5,
    0.0,
    1.0,
    450.25,
    0.12,
    3.0
]

result = predictor.predict(features)

print(result)


Writing endpoint.py


In [ ]:
!python endpoint.py